In [1]:
# Parameters
run_id = "ff65ae5c-f636-4122-9894-971278e5e4d2"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/ff65ae5c-f636-4122-9894-971278e5e4d2"
sample_size = None
epochs = None
threshold = None


# Create node embeddings feature groups.

Up until now we use feature engineering, feature store and model training to create node embedding. We will now materialise this as node embeddings feature group. This feature group will be used to train anomaly detection model.

![Feature Stores](./images/online_offline_fs.png)

---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. If live Transaction Monitoring System is based on graph or node embeddings then this will require 1st to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failures. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection live Transaction Monitoring Systems. 

---

## Query Model Repository for best node embeddings model

In [2]:
# Setup for local execution
import os
import json
import pandas as pd
import numpy as np

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")

print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")

Training data: /home/adnoman/projects/aml_gan/AMLend2end/training_data
Output: /home/adnoman/projects/aml_gan/AMLend2end/output


In [3]:
# Find the latest model directory
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('node_embeddings_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
    
    # Load metadata
    with open(os.path.join(latest_model_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    print(f"Model metrics: {metadata['metrics']}")
    print(f"Hyperparameters: {metadata['hyperparameters']}")
else:
    print("No model found! Run notebook 4 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4
Model metrics: {'accuracy': 0.8891156462585034}
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [4]:
# Load node embeddings from notebook 4
embeddings_path = os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv")
node_embeddings_df = pd.read_csv(embeddings_path)

print(f"Loaded embeddings shape: {node_embeddings_df.shape}")
node_embeddings_df.head()

Loaded embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,-0.026495,0.026853,0.021857,0.011051,-0.016078,-0.016079,0.001365,0.018063,-0.014078,...,0.022683,0.006947,0.009051,-0.023597,0.025232,0.021024,0.023285,-0.029349,-0.002892,0.000185
1,1e46e726,0.008406,-0.021091,-0.021135,0.026678,-0.015544,-0.028359,0.010794,-0.006178,-0.014040,...,0.016967,-0.009004,0.022737,0.022184,0.030982,-0.020325,-0.024437,-0.009588,0.016712,-0.004696
2,49203bc3,-0.012483,-0.021595,-0.006502,0.025198,0.007464,0.005802,0.030297,-0.011307,0.030478,...,-0.020105,-0.021394,-0.022935,-0.010670,0.020898,-0.026390,-0.025667,0.006688,-0.028166,-0.006906
3,a74d1101,0.003761,-0.012751,0.024891,0.000710,0.007301,0.026914,-0.023606,-0.006187,-0.023932,...,0.007624,0.006081,0.022695,-0.011299,-0.008927,0.023869,-0.009746,-0.015424,0.017904,0.002509
4,616d4505,-0.006480,0.020373,-0.028022,0.021887,-0.018994,-0.016227,0.015992,0.012324,0.005672,...,-0.004049,0.026130,0.005538,-0.010459,-0.011313,0.018269,0.016335,-0.006085,-0.017172,0.009438


## Define model and load wights 

In [5]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_df.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Preview embeddings
node_embeddings_df[['node_id'] + emb_cols[:5]].head()

Embedding dimensions: 32


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4
0,3aa9646b,-0.026495,0.026853,0.021857,0.011051,-0.016078
1,1e46e726,0.008406,-0.021091,-0.021135,0.026678,-0.015544
2,49203bc3,-0.012483,-0.021595,-0.006502,0.025198,0.007464
3,a74d1101,0.003761,-0.012751,0.024891,0.000710,0.007301
4,616d4505,-0.006480,0.020373,-0.028022,0.021887,-0.018994


## connect hsfs library and get fs handle

In [6]:
# Load alert nodes to join with embeddings
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))
print(f"Alert nodes: {len(alert_nodes_df)}")
print(f"SAR nodes: {alert_nodes_df['is_sar'].sum()}")

Alert nodes: 7347
SAR nodes: 816


### Get node and edge traininhg dataset objects 

In [7]:
# Create embedding array column (for compatibility with original format)
node_embeddings_df['embedding'] = node_embeddings_df[emb_cols].values.tolist()

# Rename node_id to id for consistency
node_embeddings_df = node_embeddings_df.rename(columns={'node_id': 'id'})

# Select final columns
node_embeddings_final = node_embeddings_df[['id', 'embedding']].copy()
print(f"Final embeddings shape: {node_embeddings_final.shape}")
node_embeddings_final.head()

Final embeddings shape: (7347, 2)


,id,embedding
0,3aa9646b,"[-0.026494756, 0.026853353, 0.021856649, 0.011..."
1,1e46e726,"[0.008405837, -0.02109072, -0.0211347, 0.02667..."
2,49203bc3,"[-0.012483114, -0.021595012, -0.0065024844, 0...."
3,a74d1101,"[0.0037611716, -0.012751436, 0.024891276, 0.00..."
4,616d4505,"[-0.0064802375, 0.020372668, -0.028021835, 0.0..."


### Read training datasets as pandas df 

In [8]:
# Join embeddings with alert nodes info
embeddings_with_labels = node_embeddings_df.merge(
    alert_nodes_df[['id', 'is_sar']], 
    on='id', 
    how='left'
)
embeddings_with_labels['is_sar'] = embeddings_with_labels['is_sar'].fillna(0).astype(int)

print(f"Embeddings with labels: {embeddings_with_labels.shape}")
print(f"SAR nodes in embeddings: {embeddings_with_labels['is_sar'].sum()}")

Embeddings with labels: (7347, 35)
SAR nodes in embeddings: 816


### Read hyperparamenter for graph embeddings

In [9]:
# Preview the data
print("Sample of embeddings with SAR labels:")
embeddings_with_labels[['id', 'is_sar'] + emb_cols[:3]].head(10)

Sample of embeddings with SAR labels:


,id,is_sar,emb_0,emb_1,emb_2
0,3aa9646b,0,-0.026495,0.026853,0.021857
1,1e46e726,0,0.008406,-0.021091,-0.021135
2,49203bc3,0,-0.012483,-0.021595,-0.006502
3,a74d1101,1,0.003761,-0.012751,0.024891
4,616d4505,0,-0.006480,0.020373,-0.028022
5,99af2455,1,0.026886,-0.029892,-0.023128
6,39be1ea2,0,-0.007712,0.010026,-0.021921
7,e7ec7bdb,1,-0.025434,-0.018840,0.002599
8,e2e0d938,0,-0.009638,0.009643,0.014761
9,afc399a9,0,0.020312,0.013708,0.001608


### Construct stellargraph Graph object

In [10]:
# Statistics
print("Embedding statistics:")
print(f"  Total nodes: {len(embeddings_with_labels)}")
print(f"  SAR nodes (is_sar=1): {embeddings_with_labels['is_sar'].sum()}")
print(f"  Non-SAR nodes (is_sar=0): {(embeddings_with_labels['is_sar']==0).sum()}")
print(f"  Embedding dimensions: {len(emb_cols)}")

Embedding statistics:
  Total nodes: 7347
  SAR nodes (is_sar=1): 816
  Non-SAR nodes (is_sar=0): 6531
  Embedding dimensions: 32


### infer node embeddings

In [11]:
# Prepare final feature group data
# Keep id, all embedding columns, and is_sar
final_cols = ['id'] + emb_cols + ['is_sar']
node_embeddings_fg_df = embeddings_with_labels[final_cols].copy()

print(f"Feature group shape: {node_embeddings_fg_df.shape}")
node_embeddings_fg_df.head()

Feature group shape: (7347, 34)


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,-0.026495,0.026853,0.021857,0.011051,-0.016078,-0.016079,0.001365,0.018063,-0.014078,...,0.006947,0.009051,-0.023597,0.025232,0.021024,0.023285,-0.029349,-0.002892,0.000185,0
1,1e46e726,0.008406,-0.021091,-0.021135,0.026678,-0.015544,-0.028359,0.010794,-0.006178,-0.014040,...,-0.009004,0.022737,0.022184,0.030982,-0.020325,-0.024437,-0.009588,0.016712,-0.004696,0
2,49203bc3,-0.012483,-0.021595,-0.006502,0.025198,0.007464,0.005802,0.030297,-0.011307,0.030478,...,-0.021394,-0.022935,-0.010670,0.020898,-0.026390,-0.025667,0.006688,-0.028166,-0.006906,0
3,a74d1101,0.003761,-0.012751,0.024891,0.000710,0.007301,0.026914,-0.023606,-0.006187,-0.023932,...,0.006081,0.022695,-0.011299,-0.008927,0.023869,-0.009746,-0.015424,0.017904,0.002509,1
4,616d4505,-0.006480,0.020373,-0.028022,0.021887,-0.018994,-0.016227,0.015992,0.012324,0.005672,...,0.026130,0.005538,-0.010459,-0.011313,0.018269,0.016335,-0.006085,-0.017172,0.009438,0


In [12]:
# Dummy cell - removed pyspark code

In [13]:
# Dummy cell - removed pyspark code

In [14]:
# Dummy cell - removed pyspark code

In [15]:
# Dummy cell - removed pyspark code

In [16]:
# Dummy cell - removed pyspark code

## Create embeddings feature group

In [17]:
# Save node embeddings feature group locally (replaces hsfs)
fg_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet")
node_embeddings_fg_df.to_parquet(fg_path, index=False)
print(f"Saved node embeddings feature group to: {fg_path}")

# Also save as CSV for easier inspection
csv_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.csv")
node_embeddings_fg_df.to_csv(csv_path, index=False)
print(f"Saved CSV version to: {csv_path}")

Saved node embeddings feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


Saved CSV version to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.csv


In [18]:
# Summary
print("=" * 50)
print("Node Embeddings Feature Group Created")
print("=" * 50)
print(f"Total nodes: {len(node_embeddings_fg_df)}")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"SAR nodes: {node_embeddings_fg_df['is_sar'].sum()}")
print(f"Non-SAR nodes: {(node_embeddings_fg_df['is_sar']==0).sum()}")
print(f"\nSaved to: {fg_path}")
print("=" * 50)

Node Embeddings Feature Group Created
Total nodes: 7347
Embedding dimensions: 32
SAR nodes: 816
Non-SAR nodes: 6531

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


## Feature group provenance
![Feature group provenance](./images/provenance_fg.png)

In [19]:
# Done!